# Agent-Stratified Uniform Sampling Walkthrough

This notebook demonstrates a deterministic, cost-neutral reporting prototype for tenant/agent strata.

What you will see:
- Tenant/agent stratification and per-stratum $N_a$, $n_a$, and $p_a = n_a / N_a$.
- A deterministic length-neutrality proof: changing only token cost cannot change sampled membership.
- Durable queue metadata (including seed and status) and pacing after membership is fixed.
- Oversized selected-session behavior (selected but marked `OVERSIZED`, never replaced).
- A delay-ceiling policy that marks excessively delayed selected sessions `DROPPED` without replacing them.
- Per-agent score summaries with finite-population-corrected normal-approximation intervals from completed selected sessions.

In [5]:
from __future__ import annotations

import json
import tempfile
from pathlib import Path

from agent_uniform_sampling import (
    ExecutionQueue,
    ExecutionStatus,
    SessionCandidate,
    summarize_agent_scores,
    uniformly_sample_by_agent,
)

def candidate(tenant: str, agent: str, session_id: str, tokens: int) -> SessionCandidate:
    return SessionCandidate(
        tenant_id=tenant,
        agent_id=agent,
        session_id=session_id,
        session_version="v1",
        estimated_tokens=tokens,
    )

## 1) Build a Stratified Population

We create three strata: `contoso/support`, `contoso/sales`, and `fabrikam/support`.
Each stratum has a distinct eligible population size $N_a$.

In [6]:
population = (
    candidate("contoso", "support", "cs-01", 400),
    candidate("contoso", "support", "cs-02", 650),
    candidate("contoso", "support", "cs-03", 900),
    candidate("contoso", "support", "cs-04", 2500),
    candidate("contoso", "support", "cs-05", 300),
    candidate("contoso", "sales", "sales-01", 200),
    candidate("contoso", "sales", "sales-02", 420),
    candidate("contoso", "sales", "sales-03", 1200),
    candidate("fabrikam", "support", "fs-01", 500),
    candidate("fabrikam", "support", "fs-02", 700),
    candidate("fabrikam", "support", "fs-03", 800),
    candidate("fabrikam", "support", "fs-04", 1400),
)

seed = "demo-seed-2026-08"
sample_size_per_agent = 3
samples = uniformly_sample_by_agent(
    candidates=population,
    sample_size_per_agent=sample_size_per_agent,
    seed=seed,
)

print("Per-stratum sampling parameters (N_a, n_a, p_a):")
for sample in samples:
    selected_ids = [selected.candidate.session_id for selected in sample.selected]
    print(
        f"- {sample.stratum_key}: N_a={sample.population_size}, n_a={sample.sample_size}, p_a={sample.inclusion_probability:.3f}, selected={selected_ids}"
    )

Per-stratum sampling parameters (N_a, n_a, p_a):
- contoso/sales: N_a=3, n_a=3, p_a=1.000, selected=['sales-02', 'sales-01', 'sales-03']
- contoso/support: N_a=5, n_a=3, p_a=0.600, selected=['cs-01', 'cs-03', 'cs-04']
- fabrikam/support: N_a=4, n_a=3, p_a=0.750, selected=['fs-03', 'fs-02', 'fs-04']


## 2) Deterministic Length-Neutrality Proof

We hold identity fields fixed and change only `estimated_tokens` between Case A and Case B.
If selected IDs match exactly, token cost is not part of membership.

In [7]:
def twin_population(cost_fn):
    return tuple(
        candidate("contoso", "support", f"neutral-{index:02d}", cost_fn(index))
        for index in range(10)
    )

case_a = twin_population(lambda i: 10 + i)
case_b = twin_population(lambda i: 10_000 - i)

sample_a = uniformly_sample_by_agent(candidates=case_a, sample_size_per_agent=4, seed="neutrality-seed")
sample_b = uniformly_sample_by_agent(candidates=case_b, sample_size_per_agent=4, seed="neutrality-seed")

ids_a = [item.candidate.session_id for item in sample_a[0].selected]
ids_b = [item.candidate.session_id for item in sample_b[0].selected]

assert ids_a == ids_b, "Membership changed even though only cost changed"
print("Case A selected IDs:", ids_a)
print("Case B selected IDs:", ids_b)
print("Deterministic proof passed: token cost changed, sampled membership did not.")

Case A selected IDs: ['neutral-08', 'neutral-01', 'neutral-04', 'neutral-09']
Case B selected IDs: ['neutral-08', 'neutral-01', 'neutral-04', 'neutral-09']
Deterministic proof passed: token cost changed, sampled membership did not.


## 3) Queue Metadata, Oversized/Delayed Behavior, and Per-Agent Scoring

After sampling, we enqueue selected sessions into a durable JSON queue.
- Queue metadata stores seed and per-stratum run details ($N_a$, $n_a$, $p_a$, selected request IDs).
- Scheduling respects TPM limits post-membership and marks oversized selected sessions as `OVERSIZED`.
- With `max_schedule_delay_seconds`, selected items that would be scheduled too far in the future are marked `DROPPED` (membership is retained; status changes).
- Reporting uses completed selected sessions only and returns per-agent confidence intervals when enough scores exist.

In [ ]:
tmp_queue_path = None
with tempfile.TemporaryDirectory() as tmpdir:
    queue_path = Path(tmpdir) / "queue.json"
    tmp_queue_path = queue_path
    queue = ExecutionQueue(queue_path, tpm_limit=1800)

    queue.enqueue(samples)
    items = queue.schedule_pending()

    raw = json.loads(queue_path.read_text(encoding="utf-8"))
    print("Queue keys:", sorted(raw.keys()))
    print("Sampling runs stored:", len(raw["sampling_runs"]))
    for run in sorted(raw["sampling_runs"].values(), key=lambda r: r["stratum_key"]):
        print(
            f"run={run['run_id']} stratum={run['stratum_key']} seed={run['seed']} N_a={run['population_size']} n_a={run['sample_size']} p_a={run['inclusion_probability']:.3f}"
        )

    status_counts = {}
    for item in items:
        status_counts[item.status.value] = status_counts.get(item.status.value, 0) + 1
    print("Status counts after scheduling:", status_counts)

    oversized_ids = [
        item.sampled.candidate.session_id
        for item in items
        if item.status == ExecutionStatus.OVERSIZED
    ]
    print("Oversized selected sessions (kept in sample, not replaced):", oversized_ids)

    delayed_population = (
        candidate("contoso", "support", "delay-a", 600),
        candidate("contoso", "support", "delay-b", 600),
    )
    delayed_samples = uniformly_sample_by_agent(
        candidates=delayed_population,
        sample_size_per_agent=2,
        seed="delay-ceiling-seed",
    )
    delayed_queue = ExecutionQueue(
        Path(tmpdir) / "queue-delay-ceiling.json",
        tpm_limit=1000,
        max_schedule_delay_seconds=59,
    )
    delayed_queue.enqueue(delayed_samples)
    delayed_items = delayed_queue.schedule_pending()

    delayed_selected_ids = [item.candidate.session_id for item in delayed_samples[0].selected]
    delayed_item_ids = [item.sampled.candidate.session_id for item in delayed_items]
    dropped_ids = [
        item.sampled.candidate.session_id
        for item in delayed_items
        if item.status == ExecutionStatus.DROPPED
    ]
    delayed_status_counts = {}
    for item in delayed_items:
        delayed_status_counts[item.status.value] = delayed_status_counts.get(item.status.value, 0) + 1

    assert sorted(delayed_selected_ids) == sorted(delayed_item_ids)
    print("Delay-ceiling selected IDs (membership retained):", delayed_selected_ids)
    print("Delay-ceiling status counts:", delayed_status_counts)
    print("Delay-ceiling DROPPED IDs:", dropped_ids)

    scheduled_items = [item for item in items if item.status == ExecutionStatus.SCHEDULED]
    for index, item in enumerate(scheduled_items):
        deterministic_score = ((index * 17) % 100) / 100
        queue.complete(item.request_id, score=deterministic_score)

    summaries = summarize_agent_scores(queue.items())
    print("\nPer-agent summaries (selected vs completed):")
    for summary in summaries:
        print(
            f"- {summary.tenant_id}/{summary.agent_id}: selected={summary.selected_count}, completed={summary.completed_count}, p_a={summary.inclusion_probability:.3f}, mean={summary.mean_score}, ci95={summary.confidence_interval_95}"
        )

    print("Queue exists inside temp context:", queue_path.exists())

print("Queue exists after temp cleanup:", tmp_queue_path.exists())

Queue keys: ['items', 'sampling_runs', 'schema_version', 'tpm_limit']
Sampling runs stored: 3
run=31cd3508fd9ef2b7678654d3 stratum=contoso/sales seed=demo-seed-2026-08 N_a=3 n_a=3 p_a=1.000
run=091c988457deb5ffb812899d stratum=contoso/support seed=demo-seed-2026-08 N_a=5 n_a=3 p_a=0.600
run=61786dc52c4224d4d0c37c16 stratum=fabrikam/support seed=demo-seed-2026-08 N_a=4 n_a=3 p_a=0.750
Status counts after scheduling: {'SCHEDULED': 8, 'OVERSIZED': 1}
Oversized selected sessions (kept in sample, not replaced): ['cs-04']

Per-agent summaries (selected vs completed):
- contoso/sales: selected=3, completed=3, p_a=1.000, mean=0.06999999999999999, ci95=(0.0, 0.18814318995749746)
- contoso/support: selected=3, completed=2, p_a=0.600, mean=0.595, ci95=(0.09520000000000006, 1.0)
- fabrikam/support: selected=3, completed=3, p_a=0.750, mean=0.45333333333333337, ci95=(0.1594785543844262, 0.7471881122822406)
Queue exists inside temp context: True
Queue exists after temp cleanup: False


## 4) Prototype Limits

- Reporting is agent-level only in this prototype (no fleet aggregate).
- Means and confidence intervals are based on completed selected items only.
- Oversized selected sessions remain in-sample and visible with `OVERSIZED` status; they are never replaced by shorter sessions.
- Membership remains deterministic and cost-neutral as long as identity fields and seed are unchanged.